# CentralValleySR: Notebook 04  
## Impact Analysis: Does Super-Resolution Improve Agricultural Signal Interpretation?

**Project goal in one sentence:**  
Use AI-enhanced satellite super-resolution to determine whether sharper satellite imagery improves the detection of crop-specific growth and drought-stress patterns in California's Central Valley.

The previous notebooks established three major pieces of the project:

1. **Baseline remote sensing analysis**  
   Sentinel-2, CDL crop masks, and NDVI were used to study crop-specific seasonal behavior.

2. **Super-resolution data construction**  
   Low-resolution and high-resolution image patch pairs were prepared.

3. **First machine learning model**  
   A simple CNN-based super-resolution model was trained and evaluated using reconstruction metrics such as loss and PSNR.

This notebook asks the next and most important scientific question:

> Does the super-resolution model improve the agricultural signal we care about?

That means we do not stop at asking whether the image looks sharper.  
We ask whether the super-resolved output better preserves or reconstructs **vegetation-relevant information**, especially NDVI.


By the end of this notebook, we want to produce a paper-ready comparison between:

- **LR input**: the coarse image available to the model
- **Bicubic baseline**: a standard non-ML upsampling method
- **Model prediction**: the super-resolved image produced by the CNN
- **HR target**: the best available high-resolution reference

We will compare these using:

1. **Image reconstruction metrics**
   - PSNR
   - MSE / MAE

2. **Vegetation-index metrics**
   - NDVI maps
   - NDVI error relative to HR target
   - NDVI MAE / RMSE

3. **Spatial-detail metrics**
   - edge strength
   - gradient magnitude
   - qualitative visual comparison


The hypothesis is not only that AI can make images look sharper.  
The hypothesis is that increased spatial detail can reveal crop-specific growth and stress signatures that are not clear at native resolution.

So this notebook helps support the transition from:

> “The model reconstructs images”

to:

> “The model improves the agricultural interpretation of vegetation structure and stress.”


## Phase alignment from the project timeline

This notebook corresponds to:

### Phase 4: Impact Analysis
**Purpose:** Show whether super-resolution actually improves agricultural signal interpretation.

In the timeline, this phase asks Alexander to:
- recompute NDVI using SR imagery,
- compare native NDVI vs SR NDVI,
- examine variance, seasonal clarity, and crop separability,
- identify the strongest figures for the paper.

This notebook focuses on the first part of Phase 4:  
**patch-level SR impact analysis**.

Later, this same logic can be extended to crop-level and season-level summaries.


# 0) Environment Setup

### Why this step exists
This notebook uses outputs from earlier notebooks:
- saved LR/HR patch pairs,
- the trained super-resolution model checkpoint,
- previous training results.

### What we are coding
We import libraries needed for:
- loading saved image patches,
- loading the trained model,
- computing image and NDVI metrics,
- plotting paper-style comparison figures.

### How we are coding it
We use standard tools:
- `numpy` and `pandas` for data handling,
- `matplotlib` for plotting,
- `torch` for model inference,
- optional `skimage` if available for SSIM.


In [ ]:
from pathlib import Path
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from skimage.metrics import structural_similarity as ssim
    SKIMAGE_AVAILABLE = True
except Exception:
    SKIMAGE_AVAILABLE = False

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("scikit-image available:", SKIMAGE_AVAILABLE)

# 1) Reproducibility Settings

### Why this matters
Even though this notebook focuses mostly on inference and evaluation, reproducibility still matters.  
We want the same validation patches and example figures to appear each time unless we intentionally change the seed.

### What we are coding
We set random seeds for Python, NumPy, and PyTorch.

### How we are coding it
We define a single `SEED` and apply it consistently.


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅ Seeds set to", SEED)

# 2) Define Project Paths

### Why this matters
A clean path structure keeps the notebook connected to the GitHub repository.

### What we are coding
We define expected locations for:
- super-resolution patch pairs,
- validation file list,
- trained model checkpoint,
- output figures,
- output metrics.

### How we are coding it
These paths match the GitHub skeleton and previous notebooks:
- `data_pairs/sr_pairs_x4`
- `checkpoints`
- `figures/sr_impact`
- `results`


In [ ]:
PROJECT_ROOT = Path(".").resolve()

PAIRS_DIR = PROJECT_ROOT / "data_pairs" / "sr_pairs_x4"
VAL_CSV = PAIRS_DIR / "val_files.csv"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
MODEL_PATH = CHECKPOINT_DIR / "simple_srnet_best.pt"

FIG_DIR = PROJECT_ROOT / "figures" / "sr_impact"
RESULTS_DIR = PROJECT_ROOT / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PAIRS_DIR:", PAIRS_DIR)
print("VAL_CSV exists:", VAL_CSV.exists())
print("MODEL_PATH exists:", MODEL_PATH.exists())
print("FIG_DIR:", FIG_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

# 3) Scientific Configuration: Which Bands Define NDVI?

### Why this matters
NDVI is defined as:

`NDVI = (NIR - Red) / (NIR + Red)`

For Sentinel-2:
- Red is usually **B4**
- NIR is usually **B8**

However, local arrays may store bands in different orders depending on how the earlier export was created.  
Because of that, this notebook makes band indices configurable.

### What we are coding
We define:
- `RED_IDX`: which channel corresponds to red reflectance
- `NIR_IDX`: which channel corresponds to near-infrared reflectance

### How we are coding it
For a 4-channel array, a common setup is:
- channel 0 = Blue
- channel 1 = Green
- channel 2 = Red
- channel 3 = NIR

So we start with:
- `RED_IDX = 2`
- `NIR_IDX = 3`

If the exported band order is different, this should be updated.


In [ ]:
# Edit these if the exported band order is different.
RED_IDX = 2
NIR_IDX = 3

# Scale factor from the previous SR notebooks.
SCALE = 4

# Number of validation examples to evaluate.
MAX_EVAL_PATCHES = 64

print("Using RED_IDX:", RED_IDX)
print("Using NIR_IDX:", NIR_IDX)
print("Scale factor:", SCALE)
print("Max evaluation patches:", MAX_EVAL_PATCHES)

# 4) Define the Same Model Architecture Used During Training

### Why this step exists
To load a saved PyTorch checkpoint, the architecture must match the model used during training.

### What we are coding
We redefine the simple baseline super-resolution network:
- bicubic upsampling,
- small CNN,
- residual correction.

### Why this architecture is scientifically useful
This is a defensible baseline because it asks:

> Does a learned CNN refinement improve over standard bicubic interpolation?

That is exactly the comparison a paper needs before introducing more complex models.

### How we are coding it
We use the same `SimpleSRNet` class from the training notebook.


In [ ]:
class SimpleSRNet(nn.Module):
    def __init__(self, in_channels=4, hidden_channels=64, scale=4):
        super().__init__()
        self.scale = scale

        self.conv1 = nn.Conv2d(in_channels, hidden_channels, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(hidden_channels, in_channels, kernel_size=3, padding=1)

    def forward(self, x):
        # Step 1: Upsample the LR input to HR size using bicubic interpolation.
        x_up = F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)

        # Step 2: Learn a residual correction on top of bicubic.
        h = F.relu(self.conv1(x_up))
        h = F.relu(self.conv2(h))
        h = F.relu(self.conv3(h))
        residual = self.conv4(h)

        # Step 3: Add learned correction back to bicubic result.
        out = x_up + residual

        # Step 4: Clamp to image-like reflectance range.
        out = torch.clamp(out, 0.0, 1.0)
        return out

# 5) Load Validation Patches or Create a Fallback Dataset

### Why this step exists
This notebook is designed to evaluate the trained model on held-out validation patches.  
However, if the validation files are missing during class, the notebook should still run so the evaluation logic can be taught.

### What we are coding
We define a data-loading function that:
- loads real `.npz` validation patches if available,
- otherwise creates synthetic fallback patches.

### Why the fallback is included
The fallback is **not final scientific evidence**.  
It is only a workflow safety net that lets us:
- test the code,
- understand evaluation metrics,
- generate example plots,
- avoid getting stuck on file logistics.

### How we are coding it
Each patch pair contains:
- `lr`: low-resolution patch in HWC format,
- `hr`: high-resolution target patch in HWC format.


In [ ]:
def make_demo_scene(h=512, w=512, c=4, seed=0):
    """Generate a synthetic multi-band satellite-like scene.

    This fallback mimics field-like spatial structure.
    It is not a replacement for real Sentinel-2 data.
    """
    rng = np.random.default_rng(seed)
    x = np.linspace(0, 1, w)
    y = np.linspace(0, 1, h)
    X, Y = np.meshgrid(x, y)

    base = 0.35 + 0.30 * np.sin(2 * np.pi * X) * np.cos(2 * np.pi * Y)

    blocks = np.zeros((h, w), dtype=np.float32)
    for _ in range(100):
        r0 = rng.integers(0, h - 40)
        c0 = rng.integers(0, w - 40)
        rh = rng.integers(20, 90)
        cw = rng.integers(20, 90)
        blocks[r0:r0+rh, c0:c0+cw] += rng.uniform(-0.18, 0.18)

    field = np.clip(base + blocks + 0.03 * rng.standard_normal((h, w)), 0, 1)

    blue = np.clip(field * 0.75 + 0.03 * rng.standard_normal((h, w)), 0, 1)
    green = np.clip(field * 0.85 + 0.03 * rng.standard_normal((h, w)), 0, 1)
    red = np.clip(field * 0.70 + 0.03 * rng.standard_normal((h, w)), 0, 1)
    nir = np.clip(field * 1.15 + 0.03 * rng.standard_normal((h, w)), 0, 1)

    return np.stack([blue, green, red, nir], axis=-1).astype(np.float32)


def downsample_area(hr, scale):
    """Downsample an HWC image by area averaging."""
    H, W, C = hr.shape
    assert H % scale == 0 and W % scale == 0
    return hr.reshape(H//scale, scale, W//scale, scale, C).mean(axis=(1, 3)).astype(np.float32)


def extract_demo_patches(scene, hr_patch=64, stride=64, scale=4, max_patches=64):
    """Extract demo HR patches and create LR patches by downsampling."""
    H, W, C = scene.shape
    pairs = []
    for r in range(0, H - hr_patch + 1, stride):
        for c in range(0, W - hr_patch + 1, stride):
            hr = scene[r:r+hr_patch, c:c+hr_patch, :]
            lr = downsample_area(hr, scale)
            pairs.append((lr, hr))
            if len(pairs) >= max_patches:
                return pairs
    return pairs


def load_validation_pairs(max_eval=64):
    """Load validation LR/HR patch pairs.

    Returns:
        pairs: list of (lr_hwc, hr_hwc)
        source_label: string describing whether real or fallback data were used
    """
    if VAL_CSV.exists() and PAIRS_DIR.exists():
        files = pd.read_csv(VAL_CSV)["file"].tolist()
        files = files[:max_eval]

        pairs = []
        for fname in files:
            path = PAIRS_DIR / fname
            if path.exists():
                d = np.load(path)
                pairs.append((d["lr"].astype(np.float32), d["hr"].astype(np.float32)))

        if len(pairs) > 0:
            return pairs, "real validation patches"

    scene = make_demo_scene(seed=SEED)
    pairs = extract_demo_patches(scene, hr_patch=64, stride=64, scale=SCALE, max_patches=max_eval)
    return pairs, "synthetic fallback patches"


pairs, source_label = load_validation_pairs(MAX_EVAL_PATCHES)

print("Loaded:", len(pairs), "pairs from", source_label)
print("Example LR shape:", pairs[0][0].shape)
print("Example HR shape:", pairs[0][1].shape)

# 6) Determine the Number of Channels and Load the Model

### Why this step exists
The number of channels must match between:
- the validation data,
- the model architecture,
- the saved checkpoint.

### What we are coding
We:
1. inspect the example HR patch,
2. create a model with the correct number of input channels,
3. load the trained checkpoint if available.

### What happens if the checkpoint is missing?
If the checkpoint is missing, we still run the evaluation workflow using a randomly initialized model.  
That result is **not scientifically meaningful**, but it lets the notebook run and demonstrates the logic.

### How we are coding it
We use `torch.load` only if `MODEL_PATH` exists.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

example_lr, example_hr = pairs[0]
IN_CHANNELS = example_hr.shape[-1]

model = SimpleSRNet(in_channels=IN_CHANNELS, hidden_channels=64, scale=SCALE).to(device)

if MODEL_PATH.exists():
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print("✅ Loaded trained model checkpoint from:", MODEL_PATH)
else:
    print("⚠️ No model checkpoint found. Using randomly initialized model.")
    print("This is only useful for testing the notebook, not for final results.")

model.eval()
print("Device:", device)
print("Input channels:", IN_CHANNELS)

# 7) Utility Functions for Tensor Conversion and Visualization

### Why this step exists
The data are stored in HWC format, but PyTorch models expect CHW format.

### What we are coding
Helper functions to:
- convert HWC arrays to tensors,
- convert tensors back to HWC arrays,
- safely clip values for display.

### How we are coding it
These helper functions prevent repetitive and error-prone reshaping code.


In [ ]:
def hwc_to_tensor(x):
    """Convert HWC numpy array to BCHW torch tensor."""
    x = np.transpose(x, (2, 0, 1))  # HWC -> CHW
    x = torch.from_numpy(x).float().unsqueeze(0)  # CHW -> BCHW
    return x


def tensor_to_hwc(x):
    """Convert BCHW or CHW tensor to HWC numpy array."""
    if x.ndim == 4:
        x = x.squeeze(0)
    arr = x.detach().cpu().numpy()
    arr = np.transpose(arr, (1, 2, 0))
    return arr


def safe_display(x):
    """Clip array to 0 to 1 for display."""
    return np.clip(x, 0, 1)

# 8) NDVI Function

### Why NDVI matters here
The goal is not simply to produce prettier images.  
The project asks whether super-resolution improves agricultural interpretation.

NDVI is one of the clearest agricultural signals because it reflects vegetation greenness and canopy activity.

### What we are coding
We compute NDVI from image arrays using the configured red and NIR band indices.

### How we are coding it
We add a small epsilon to the denominator to avoid division-by-zero errors.


In [ ]:
def compute_ndvi_hwc(img_hwc, red_idx=RED_IDX, nir_idx=NIR_IDX, eps=1e-6):
    """Compute NDVI from an HWC image array."""
    red = img_hwc[..., red_idx]
    nir = img_hwc[..., nir_idx]
    ndvi = (nir - red) / (nir + red + eps)
    return np.clip(ndvi, -1, 1)


test_ndvi = compute_ndvi_hwc(example_hr)
print("NDVI shape:", test_ndvi.shape)
print("NDVI min/max:", np.nanmin(test_ndvi), np.nanmax(test_ndvi))

# 9) Image and NDVI Metrics

### Why we need metrics
Visual inspection is important, but a paper also needs quantitative evidence.

### What we are coding
We compute several patch-level metrics:

#### Image-level metrics
- MSE: mean squared error
- MAE: mean absolute error
- PSNR: standard reconstruction metric

#### NDVI-level metrics
- NDVI MAE
- NDVI RMSE

#### Spatial-detail metric
- mean gradient magnitude, which approximates edge strength and fine spatial variation

### How we are coding it
Each metric is a small function that takes NumPy arrays.


In [ ]:
def mse(a, b):
    return float(np.mean((a - b) ** 2))


def mae(a, b):
    return float(np.mean(np.abs(a - b)))


def psnr(a, b, max_val=1.0):
    value = mse(a, b)
    if value == 0:
        return float("inf")
    return 20 * math.log10(max_val) - 10 * math.log10(value)


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def gradient_magnitude_mean(img_2d):
    """Mean gradient magnitude for a 2D image or NDVI map."""
    gy, gx = np.gradient(img_2d)
    grad = np.sqrt(gx**2 + gy**2)
    return float(np.mean(grad))


def optional_ssim(a, b):
    """Compute SSIM if scikit-image is available. Returns NaN otherwise."""
    if not SKIMAGE_AVAILABLE:
        return np.nan
    try:
        return float(ssim(a, b, data_range=1.0, channel_axis=-1))
    except Exception:
        return np.nan

# 10) Run Model Inference on Validation Patches

### Why this step exists
This is where we generate the actual super-resolution outputs.

For each validation pair, we compute:
1. bicubic upsampling of LR,
2. model prediction,
3. HR target comparison.

### What we are coding
We loop over validation patches and save:
- LR,
- bicubic,
- model prediction,
- HR target,
- NDVI maps,
- metrics.

### How we are coding it
The model runs in `torch.no_grad()` mode because we are evaluating, not training.


In [ ]:
records = []
examples = []

with torch.no_grad():
    for idx, (lr_hwc, hr_hwc) in enumerate(pairs):
        lr_tensor = hwc_to_tensor(lr_hwc).to(device)

        pred_tensor = model(lr_tensor)

        lr_up_tensor = F.interpolate(
            lr_tensor,
            size=hr_hwc.shape[:2],
            mode="bicubic",
            align_corners=False
        )

        pred_hwc = safe_display(tensor_to_hwc(pred_tensor))
        bicubic_hwc = safe_display(tensor_to_hwc(lr_up_tensor))
        hr_hwc = safe_display(hr_hwc)
        lr_hwc = safe_display(lr_hwc)

        ndvi_hr = compute_ndvi_hwc(hr_hwc)
        ndvi_bicubic = compute_ndvi_hwc(bicubic_hwc)
        ndvi_pred = compute_ndvi_hwc(pred_hwc)

        records.append({
            "patch_id": idx,
            "image_mae_bicubic": mae(bicubic_hwc, hr_hwc),
            "image_mae_model": mae(pred_hwc, hr_hwc),
            "image_mse_bicubic": mse(bicubic_hwc, hr_hwc),
            "image_mse_model": mse(pred_hwc, hr_hwc),
            "image_psnr_bicubic": psnr(bicubic_hwc, hr_hwc),
            "image_psnr_model": psnr(pred_hwc, hr_hwc),
            "image_ssim_bicubic": optional_ssim(bicubic_hwc, hr_hwc),
            "image_ssim_model": optional_ssim(pred_hwc, hr_hwc),
            "ndvi_mae_bicubic": mae(ndvi_bicubic, ndvi_hr),
            "ndvi_mae_model": mae(ndvi_pred, ndvi_hr),
            "ndvi_rmse_bicubic": rmse(ndvi_bicubic, ndvi_hr),
            "ndvi_rmse_model": rmse(ndvi_pred, ndvi_hr),
            "ndvi_gradient_hr": gradient_magnitude_mean(ndvi_hr),
            "ndvi_gradient_bicubic": gradient_magnitude_mean(ndvi_bicubic),
            "ndvi_gradient_model": gradient_magnitude_mean(ndvi_pred),
        })

        if len(examples) < 5:
            examples.append({
                "lr": lr_hwc,
                "bicubic": bicubic_hwc,
                "pred": pred_hwc,
                "hr": hr_hwc,
                "ndvi_bicubic": ndvi_bicubic,
                "ndvi_pred": ndvi_pred,
                "ndvi_hr": ndvi_hr,
            })

metrics_df = pd.DataFrame(records)
display(metrics_df.head())

print("Evaluated patches:", len(metrics_df))

# 11) Summarize Metrics Across Validation Patches

### Why this matters
A single patch can be misleading.  
We need aggregate metrics across the validation set.

### What we are coding
We compute mean values for:
- bicubic baseline,
- model prediction,
- improvement from bicubic to model.

### How we are coding it
We compute a compact summary table that can be saved and referenced in the paper.


In [ ]:
summary = {
    "metric": [],
    "bicubic_mean": [],
    "model_mean": [],
    "model_minus_bicubic": [],
}

metric_pairs = [
    ("Image MAE", "image_mae_bicubic", "image_mae_model"),
    ("Image MSE", "image_mse_bicubic", "image_mse_model"),
    ("Image PSNR", "image_psnr_bicubic", "image_psnr_model"),
    ("Image SSIM", "image_ssim_bicubic", "image_ssim_model"),
    ("NDVI MAE", "ndvi_mae_bicubic", "ndvi_mae_model"),
    ("NDVI RMSE", "ndvi_rmse_bicubic", "ndvi_rmse_model"),
]

for label, bic_col, mod_col in metric_pairs:
    bic_mean = metrics_df[bic_col].mean()
    mod_mean = metrics_df[mod_col].mean()
    summary["metric"].append(label)
    summary["bicubic_mean"].append(bic_mean)
    summary["model_mean"].append(mod_mean)
    summary["model_minus_bicubic"].append(mod_mean - bic_mean)

summary_df = pd.DataFrame(summary)
display(summary_df)

out_path = RESULTS_DIR / "sr_impact_metrics_summary.csv"
summary_df.to_csv(out_path, index=False)
print("✅ Saved summary metrics to:", out_path)

# 12) Interpret the Metrics Correctly

### Why interpretation matters
Not all metrics should move in the same direction.

For error metrics:
- lower is better
- examples: MAE, MSE, RMSE

For reconstruction quality:
- higher is better
- examples: PSNR, SSIM

For edge or gradient metrics:
- interpretation is more subtle
- too low means oversmoothing
- too high may mean noise or artifacts

### What we are coding
We create a short helper summary that states whether the model improved relative to bicubic for each metric.

### How we are coding it
We compare model mean vs bicubic mean using the correct direction for each metric.


In [ ]:
def improvement_label(metric_name, bicubic_value, model_value):
    lower_better = ["MAE", "MSE", "RMSE"]
    higher_better = ["PSNR", "SSIM"]

    if any(x in metric_name for x in lower_better):
        return "improved" if model_value < bicubic_value else "not improved"
    if any(x in metric_name for x in higher_better):
        return "improved" if model_value > bicubic_value else "not improved"
    return "interpret carefully"

summary_df["interpretation"] = [
    improvement_label(row["metric"], row["bicubic_mean"], row["model_mean"])
    for _, row in summary_df.iterrows()
]

display(summary_df)

# 13) Visual Comparison Figure: RGB-like Image Patches

### Why this figure matters
This is the figure that helps readers visually understand what the model is doing.

### What we are coding
For selected validation examples, we show:
1. LR input
2. bicubic baseline
3. model prediction
4. HR target

### How we are coding it
Because the arrays may contain 4 bands, we display the first three channels as an RGB-like composite.  
If the channels are not true RGB, the figure should be labeled as a **false-color or band-composite visualization**.

### Paper note
This figure should be described honestly:
- If the model is smoother than HR, say so.
- If the model sharpens boundaries relative to bicubic, say so.
- If artifacts appear, acknowledge them.


In [ ]:
def display_rgb_like(img_hwc):
    """Return first three channels for visualization."""
    if img_hwc.shape[-1] >= 3:
        return safe_display(img_hwc[..., :3])
    return safe_display(np.repeat(img_hwc[..., :1], 3, axis=-1))


for i, ex in enumerate(examples[:3]):
    plt.figure(figsize=(12, 3))

    titles = ["LR input", "Bicubic", "Model prediction", "HR target"]
    imgs = [ex["lr"], ex["bicubic"], ex["pred"], ex["hr"]]

    for j, (title, img) in enumerate(zip(titles, imgs), start=1):
        plt.subplot(1, 4, j)
        plt.imshow(display_rgb_like(img))
        plt.title(title)
        plt.axis("off")

    plt.suptitle(f"SR visual comparison: patch {i}")
    plt.tight_layout()

    save_path = FIG_DIR / f"sr_visual_comparison_patch_{i}.png"
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

# 14) Visual Comparison Figure: NDVI Maps

### Why this figure is scientifically important
The RGB-like comparison tells us whether images look sharper.  
The NDVI comparison tells us whether the model preserves vegetation-relevant information.

### What we are coding
For selected patches, we show:
1. NDVI from bicubic image
2. NDVI from model prediction
3. NDVI from HR target
4. absolute NDVI error for the model

### How we are coding it
We use the configured red and NIR channels to compute NDVI for each output.


In [ ]:
for i, ex in enumerate(examples[:3]):
    ndvi_error = np.abs(ex["ndvi_pred"] - ex["ndvi_hr"])

    plt.figure(figsize=(12, 3))

    plt.subplot(1, 4, 1)
    plt.imshow(ex["ndvi_bicubic"], vmin=-1, vmax=1)
    plt.title("NDVI Bicubic")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(ex["ndvi_pred"], vmin=-1, vmax=1)
    plt.title("NDVI Model")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(ex["ndvi_hr"], vmin=-1, vmax=1)
    plt.title("NDVI HR Target")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(ndvi_error, vmin=0, vmax=max(0.5, np.nanmax(ndvi_error)))
    plt.title("|NDVI Error|")
    plt.axis("off")

    plt.suptitle(f"NDVI impact comparison: patch {i}")
    plt.tight_layout()

    save_path = FIG_DIR / f"ndvi_impact_comparison_patch_{i}.png"
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

# 15) Spatial Detail: Gradient Analysis

### Why edge or gradient analysis matters
One claim of super-resolution is that it can recover sharper spatial structure.

For agriculture, this may correspond to:
- field boundaries,
- row structure,
- within-field heterogeneity,
- canopy texture.

### What we are coding
We compare the mean NDVI gradient magnitude for:
- HR target,
- bicubic baseline,
- model prediction.

### How to interpret this
If bicubic has much lower gradient than HR, it is oversmoothing.  
If the model gradient moves closer to HR, the model may be recovering spatial detail.  
If the model gradient is much higher than HR, it may be adding artifacts or noise.


In [ ]:
gradient_summary = metrics_df[[
    "ndvi_gradient_hr",
    "ndvi_gradient_bicubic",
    "ndvi_gradient_model"
]].mean().reset_index()

gradient_summary.columns = ["source", "mean_gradient"]
display(gradient_summary)

plt.figure()
plt.bar(gradient_summary["source"], gradient_summary["mean_gradient"])
plt.ylabel("Mean NDVI gradient magnitude")
plt.title("Spatial-detail comparison using NDVI gradients")
plt.xticks(rotation=20)
plt.tight_layout()

save_path = FIG_DIR / "ndvi_gradient_summary.png"
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

# 16) Create a Paper-Ready Results Paragraph Draft

### Why this step exists
One of the goals of the project is not just to code, but to turn code into a publishable research paper.

### What we are coding
We automatically generate a short draft paragraph using the computed metrics.

### How we are coding it
The paragraph is only a starting point.  
Alexander should edit it based on:
- the actual figures,
- whether the model truly improved,
- whether outputs are real data or fallback data.


In [ ]:
def get_metric_value(metric_label, column):
    row = summary_df[summary_df["metric"] == metric_label]
    if len(row) == 0:
        return np.nan
    return float(row[column].iloc[0])

bic_psnr = get_metric_value("Image PSNR", "bicubic_mean")
mod_psnr = get_metric_value("Image PSNR", "model_mean")
bic_ndvi_rmse = get_metric_value("NDVI RMSE", "bicubic_mean")
mod_ndvi_rmse = get_metric_value("NDVI RMSE", "model_mean")

paragraph = f"""
The super-resolution model was evaluated against a bicubic upsampling baseline using held-out validation patches.
Across the evaluated patches, the bicubic baseline achieved a mean PSNR of {bic_psnr:.2f} dB, while the model achieved a mean PSNR of {mod_psnr:.2f} dB.
To assess whether image reconstruction translated into vegetation-relevant improvement, NDVI was recomputed from the bicubic, model-predicted, and HR target patches.
The bicubic NDVI RMSE was {bic_ndvi_rmse:.4f}, compared to {mod_ndvi_rmse:.4f} for the model prediction.
These results provide an initial test of whether learned super-resolution improves not only visual reconstruction but also downstream agricultural signal recovery.
"""

print(paragraph)

# 17) Save Full Patch-Level Metrics

### Why this matters
The summary table is useful for the paper, but the full patch-level table is useful for:
- debugging,
- making boxplots later,
- comparing crops or seasons later,
- preserving reproducibility.

### What we are coding
We save:
- full patch-level metrics,
- summary-level metrics.

### How we are coding it
Pandas CSV export into the `results/` folder.


In [ ]:
metrics_path = RESULTS_DIR / "sr_impact_patch_metrics.csv"
summary_path = RESULTS_DIR / "sr_impact_metrics_summary_with_interpretation.csv"

metrics_df.to_csv(metrics_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("✅ Saved patch-level metrics to:", metrics_path)
print("✅ Saved summary metrics to:", summary_path)

# 18) Write-to-Paper Prompts

Answer prompts immediately after running the notebook.

## Results: Super-Resolution Impact

1. Did the model improve PSNR relative to bicubic?
2. Did the model improve NDVI RMSE or NDVI MAE relative to bicubic?
3. Did the model preserve field boundaries or spatial detail better than bicubic?
4. Did the model introduce artifacts?
5. Are the results strong enough to support the claim that SR improves agricultural interpretation?

## Suggested paragraph structure to be placed in your current paper draft:

**Paragraph 1: Image reconstruction results**  
Describe PSNR, MAE, and visual comparison results.

**Paragraph 2: NDVI impact results**  
Describe NDVI RMSE/MAE and whether the SR output better matched the HR target.

**Paragraph 3: Scientific interpretation**  
Connect results back to crop monitoring, spatial detail, and drought-stress detection.

## Important limitation statement regarding the current notebook infrastructure:

If this notebook was run using synthetic fallback patches, the results should **not** be treated as final scientific results.

For final paper results, rerun this notebook using real validation patches generated from Sentinel-2 imagery over the study AOI!!!
